# Lesson 4.4 — Validation, Retry & Feedback Loops for Extraction Quality

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Append specific validation errors to the prompt on retry to guide correction
- Identify when retries help (format/structure) vs when they can't (info absent)
- Add detected_pattern fields to analyze false-positive dismissal patterns
- Design self-correction checks (calculated vs stated total, conflict_detected)

**You will produce:** A validation-retry loop with error feedback and a calculated-vs-stated total check.

## Concepts

**Retry-with-error-feedback:** on validation failure, send a follow-up including the original document, the failed extraction, and the **specific validation errors** to guide correction.

**Limits of retry:** retries fix **format mismatches** and **structural output errors** but **cannot** conjure information that is simply **absent** from the source (e.g. a value that exists only in an external document). Detect that case and stop retrying.

**Semantic vs syntax errors:** tool use removes syntax errors; semantic errors (values don't sum, wrong field placement) need explicit checks. Design **self-correction validation**: extract `calculated_total` alongside `stated_total` to flag discrepancies, and add `conflict_detected` booleans for inconsistent source data.

**Feedback loops:** add a `detected_pattern` field to findings so you can analyze which constructs trigger dismissed false positives.

> **Reference:** [Anthropic - increase output consistency](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Validate and build an error-feedback retry prompt

Validation yields specific errors we feed back verbatim on the next attempt.

In [ ]:
def validate(extraction):
    errors = []
    if extraction.get("total") is None:
        errors.append("field 'total' is required but null")
    items = extraction.get("line_items", [])
    calc = sum(i.get("amount", 0) for i in items)
    if extraction.get("total") not in (None, calc):
        errors.append(f"line items sum to {calc} but total is {extraction['total']}")
    return errors

def retry_prompt(doc, failed, errors):
    return (f"DOCUMENT:\n{doc}\n\nFAILED EXTRACTION:\n{failed}\n\n"
            f"VALIDATION ERRORS:\n- " + "\n- ".join(errors) +
            "\n\nReturn a corrected extraction.")

bad = {"total": 100, "line_items": [{"amount": 60}, {"amount": 30}]}
errs = validate(bad)
print("errors:", errs)
print(retry_prompt("...doc...", bad, errs)[:200])

## Code-Along: Step 2 — Decide if a retry can succeed

Retry format/structure errors; stop when the info is absent from the source.

In [ ]:
def retry_will_help(error_type):
    return error_type in {"format_mismatch", "structural_output", "sum_mismatch"}

for et in ["format_mismatch", "info_absent_in_source", "sum_mismatch"]:
    print(f"{et}: retry {'helps' if retry_will_help(et) else 'will NOT help'}")

## Code-Along: Step 3 — Self-correction: calculated vs stated total

Extract both values and flag a conflict instead of silently trusting one.

In [ ]:
def reconcile(stated_total, line_items):
    calculated = sum(i['amount'] for i in line_items)
    return {
        "stated_total": stated_total,
        "calculated_total": calculated,
        "conflict_detected": stated_total != calculated,
    }

print(reconcile(100, [{"amount": 60}, {"amount": 30}]))
print(reconcile(90, [{"amount": 60}, {"amount": 30}]))

## Exercise

Build `extract_with_retry(doc, extract_fn, max_retries=2)` that validates, and on failure re-calls `extract_fn` with a `retry_prompt`, but **short-circuits** when the error type is `info_absent_in_source`. Add a `detected_pattern` field to each finding and tally which patterns were dismissed most.

In [ ]:
# TODO: implement extract_with_retry with error feedback and absent-info short-circuit
# TODO: attach detected_pattern to findings and count dismissals per pattern

## Key Takeaways

- On failure, feed back the document, failed extraction, and specific errors for correction.
- Retries fix format/structure errors but cannot supply info absent from the source.
- Validate semantics explicitly (calculated vs stated total; conflict_detected).
- Add detected_pattern fields to analyze false-positive dismissal patterns.

---

**Next:** [Lesson 4.5 — Efficient Batch Processing Strategies](./lesson-4.5-batch-processing.ipynb)